# 01 - Fundamentos de Gradient Boosting y XGBoost en Regresión

## Curso de Machine Learning: Modelos de Regresión

Mientras que **Bagging (Random Forest)** entrena múltiples árboles profundos e independientes en paralelo para promediar y reducir la varianza, los métodos de **Boosting** entrenan una secuencia de modelos débiles (*weak learners*, típicamente árboles poco profundos) en serie, donde cada nuevo árbol se enfoca específicamente en **corregir los errores residuales** cometidos por los árboles anteriores.

**XGBoost (Extreme Gradient Boosting - Tianqi Chen & Carlos Guestrin, 2016)** es la implementación de Gradient Boosting más utilizada en ciencia de datos aplicada debido a su velocidad computacional extrema, su manejo formal de aproximaciones de Taylor de segundo orden y su regularización integrada.

---

### Contenidos
1. Filosofía de Boosting vs. Bagging (Reducción de Sesgo vs. Varianza)
2. Descenso de Gradiente en el Espacio de Funciones
3. Formulación Matemática de XGBoost: Gradientes, Hessianos y Regularización
4. Ajuste Práctico con `xgboost.XGBRegressor` en el Dataset Auto MPG
5. Monitoreo de Curvas de Aprendizaje (Pérdida Train vs. Validación)
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print(f"Entorno configurado. XGBoost version: {xgb.__version__}")


## 1. Descenso de Gradiente en el Espacio de Funciones

En regresión, el objetivo es minimizar una función de pérdida $\mathcal{L}(y, \hat{y})$ (por ejemplo, el error cuadrático $\frac{1}{2}(y - \hat{y})^2$):

1. **Paso Inicial ($m=0$)**: Se inicia con un valor constante óptimo:
   $$\hat{y}_i^{(0)} = \bar{y} = \frac{1}{N} \sum_{i=1}^N y_i$$

2. **Iteraciones ($m = 1, 2, \dots, M$)**:
   - Se calculan los **pseudo-residuos** (el gradiente negativo de la pérdida):
     $$r_{im} = - \left[ \frac{\partial \mathcal{L}(y_i, \hat{y}_i)}{\partial \hat{y}_i} \right]_{\hat{y} = \hat{y}^{(m-1)}} = y_i - \hat{y}_i^{(m-1)}$$
   - Se entrena un árbol base débil $h_m(\mathbf{x})$ para predecir estos residuos $r_{im}$.
   - Se actualiza la predicción global aplicando una **tasa de aprendizaje (shrinkage / learning rate $\eta \in (0, 1]$)**:
     $$\hat{y}_i^{(m)} = \hat{y}_i^{(m-1)} + \eta \cdot h_m(\mathbf{x}_i)$$


## 2. Las Mejoras Matemáticas de XGBoost

A diferencia del Gradient Boosting clásico, XGBoost:
1. **Aproximación de Taylor de 2º Orden**: Emplea tanto la primera derivada (**gradiente** $g_i$) como la segunda derivada (**hessiano** $h_i$) de la función de pérdida:
   $$\mathcal{L}^{(m)} \approx \sum_{i=1}^N \left[ \mathcal{L}(y_i, \hat{y}_i^{(m-1)}) + g_i f_m(\mathbf{x}_i) + \frac{1}{2} h_i f_m^2(\mathbf{x}_i) \right] + \Omega(f_m)$$

2. **Término de Regularización Analítica $\Omega(f)$**:
   $$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2 + \alpha \sum_{j=1}^T |w_j|$$
   Donde $T$ es el número de hojas y $w_j$ los pesos de salida. Esto penaliza directamente árboles sobrecomplejos.


In [ ]:
# Carga de datos Auto MPG
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Ajuste con XGBRegressor monitoreando la perdida iteracion a iteracion
model_xgb = xgb.XGBRegressor(
    n_estimators=120,
    learning_rate=0.08,
    max_depth=4,
    random_state=42,
    eval_metric='rmse'
)

# Monitoreo en train y validacion
eval_set = [(X_tr, y_tr), (X_val, y_val)]
model_xgb.fit(X_tr, y_tr, eval_set=eval_set, verbose=False)

results = model_xgb.evals_result()
train_rmse = results['validation_0']['rmse']
val_rmse = results['validation_1']['rmse']

# Grafico de Curvas de Aprendizaje
plt.figure(figsize=(10, 4.8))
plt.plot(train_rmse, label='RMSE Entrenamiento', color='#2b5c8f', lw=2)
plt.plot(val_rmse, label='RMSE Validacion', color='#d95f02', lw=2)
plt.xlabel('Numero de Arboles de Boosting (Iteraciones)', fontweight='bold')
plt.ylabel('RMSE (mpg)', fontweight='bold')
plt.title('Curvas de Aprendizaje en XGBoost: Descenso de Error Residuo a Residuo', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# Evaluacion en conjunto de prueba independiente
y_pred_xgb = model_xgb.predict(X_test)
print(f"R2 Final en Prueba:  {r2_score(y_test, y_pred_xgb):.4f}")
print(f"RMSE Final en Prueba: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.2f} mpg")


## 3. Conclusiones

1. **Ajuste a Residuos**: Cada árbol nuevo se enfoca exclusivamente en la variabilidad no explicada por el ensamble previo.
2. **Learning Rate ($\eta$)**: Valores moderados ($0.03 - 0.1$) desaceleran el aprendizaje pero alcanzan soluciones con menor error de generalización.
3. **Regularización**: La penalización de la complejidad de las hojas en XGBoost previene el sobreajuste inherente al boosting tradicional.

En el siguiente cuaderno, **02_Hiperparametros_Regularizacion_y_Early_Stopping.ipynb**, estudiaremos cómo calibrar los hiperparámetros y aplicar **Early Stopping** para optimizar el punto de parada automático.
